# License
This notebook is licensed under the MIT License. See the LICENSE file in the repository root for details.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q transformers faker pandas seaborn matplotlib
    print('Packages installed.')


# Qualitative Analysis with LLMs (Zero-Shot Classification) In-Depth

Traditional qualitative coding is time-consuming. Large Language Models (LLMs) can automate the initial coding pass using Zero-Shot Classification. This notebook demonstrates how to categorize synthetic interview transcripts into themes without training a model.

**Techniques:**
1.  **Synthetic Interview Generation**: Using `Faker` to simulate qualitative data.
2.  **Zero-Shot Classification**: Using Hugging Face `transformers` to assign themes.
3.  **Visualization**: Comparing predicted themes.

In [ ]:
import pandas as pd
import random
from faker import Faker
from transformers import pipeline
import seaborn as sns
import matplotlib.pyplot as plt

# specific model for zero-shot
model_name = "facebook/bart-large-mnli"

## 1. Generating Synthetic Interview Data
We simulate responses from employees about their job satisfaction. We inject specific keywords to represent latent themes: `Career Growth`, `Work-Life Balance`, and `Company Culture`.

In [ ]:
fake = Faker()
Faker.seed(42)
random.seed(42)

def generate_interview_data(n=20):
    topics = {
        "Career Growth": ["promotion", "salary", "learning", "skills", "manager", "climb", "ladder"],
        "Work-Life Balance": ["overtime", "weekend", "family", "stress", "burnout", "hours", "relax"],
        "Company Culture": ["team", "collaboration", "toxic", "supportive", "values", "environment", "peers"]
    }
    
    data = []
    for _ in range(n):
        # Choose a random topic
        topic = random.choice(list(topics.keys()))
        keywords = topics[topic]
        
        # Generate a base sentence
        # We'll construct a sentence that sounds like feedback
        prefix = random.choice(["I feel that", "My experience is that", "Honestly,", "To be fair,"])
        context = fake.sentence(nb_words=6).replace('.', '')
        
        # Inject 1-2 keywords naturally (simplified for demo)
        k1 = random.choice(keywords)
        k2 = random.choice(keywords)
        
        text = f"{prefix} {context} regarding my {k1} and {k2}."
        
        data.append({"response_id": fake.uuid4()[:8], "text": text, "true_theme": topic})
    
    return pd.DataFrame(data)

df = generate_interview_data(10)
print(df[['text', 'true_theme']].head())

## 2. Zero-Shot Classification
We use a pre-trained NLI (Natural Language Inference) model. We provide the "candidate labels" (our codes), and the model determines which label best entails the text.

In [ ]:
# Initialize the pipeline (downloads model on first run)
classifier = pipeline("zero-shot-classification", model=model_name)

candidate_labels = ["Career Growth", "Work-Life Balance", "Company Culture"]

def classify_response(text):
    result = classifier(text, candidate_labels)
    # Return the top label
    return result['labels'][0]

print("Classifying responses... This may take a moment on CPU.")
df['predicted_theme'] = df['text'].apply(classify_response)

print(df[['text', 'true_theme', 'predicted_theme']].head())

## 3. Results Visualization
Compare the distribution of themes.

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(x='predicted_theme', data=df, palette='viridis')
plt.title('Distribution of Predicted Themes')
plt.xlabel('Theme')
plt.ylabel('Count')
plt.show()

## 4. Accuracy Check
Since we generated the data with intent, we can check how well the Zero-Shot model recovered our "true" themes.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

print(classification_report(df['true_theme'], df['predicted_theme']))

cm = confusion_matrix(df['true_theme'], df['predicted_theme'], labels=candidate_labels)
sns.heatmap(cm, annot=True, fmt='d', xticklabels=candidate_labels, yticklabels=candidate_labels, cmap='Blues')
plt.ylabel('True Theme')
plt.xlabel('Predicted Theme')
plt.title('Confusion Matrix')
plt.show()